# 09 - Run a bounded angle sequence


Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


**Physical motion:** inspect cable slack, attached optics, and the complete possible
rotation path first. Existing direction/backlash settings determine the route; a small
angle difference does not guarantee a short path. Home/move may enable a disabled motor.
The example does not automatically home, retry, or return to zero.

Calls block until completion. Interrupting the cell normally delivers `KeyboardInterrupt`,
which triggers a profiled-stop attempt on the owning connection. Kernel shutdown, force
termination, a lost USB link, or interrupt delivery failure cannot guarantee a stop.
Use your apparatus's physical safety procedure if control is lost. A controller-reported
stop is not independent proof of mechanical rest. Closing USB leaves motor holding state
unchanged.


In [ ]:
from dataclasses import asdict

from k10cr2 import K10CR2

SERIAL = ""  # Copy the serial from discovery and verify it on the mount's label.
CONNECT_TO_DEVICE = False  # Set True only when ready for USB access.


def configured_mount(**settings):
    # Construction validates the serial and timeouts but does not open USB.
    return K10CR2(SERIAL, **settings)

In [ ]:
from k10cr2 import MotionError


def report_motion_error(error):
    print("Original failure:", type(error.original).__name__, str(error.original))
    print("Stop cleanup:", asdict(error.cleanup))
    if not error.cleanup.confirmed_stopped:
        print("PHYSICAL STATE UNKNOWN: follow the apparatus safety procedure.")
    print("Close and explicitly reconnect; inspect state before deciding on another motion.")

## Validate the whole target list before connection

Use a short, apparatus-specific list. These targets are examples, not a route recommendation.
Each move is absolute; no automatic home or final return is included. A future measurement
can be inserted only after the blocking move returns. That ordering establishes controller
completion, not optical settling or independent mechanical-rest verification.


In [ ]:
from k10cr2.settings import resolve_target

TARGETS_DEG = [5.0, 10.0, 0.0]
if not 1 <= len(TARGETS_DEG) <= 20:
    raise ValueError("Choose between 1 and 20 explicit targets for this tutorial.")
preview = [{"target_deg": target, "counts": resolve_target(target, 0)} for target in TARGETS_DEG]
for row in preview:
    print(row)

## One owner, sequential moves, no retries

Enter `SEQUENCE ` followed by the serial after reviewing **every** target and its possible
route. Validate targets again in this cell so an edit cannot skip full-list validation.
The first failure stops the sequence; subsequent targets are not attempted. Successful
records remain in `sequence_results` for inspection. No record is saved to disk here.


In [ ]:
CONFIRM_SEQUENCE = ""  # Type "SEQUENCE " followed by the actual serial to opt in.
sequence_results = []

if CONNECT_TO_DEVICE and SERIAL and CONFIRM_SEQUENCE == f"SEQUENCE {SERIAL}":
    if not 1 <= len(TARGETS_DEG) <= 20:
        raise ValueError("Choose between 1 and 20 explicit targets.")
    validated_targets = tuple(TARGETS_DEG)
    for target in validated_targets:
        resolve_target(target, 0)
    with configured_mount() as mount:
        state = mount.status()
        if state.busy or not state.homed or state.motion_error or state.raw_bits & 0x40000000:
            raise RuntimeError("Sequence requires a homed, idle controller with no error flags.")
        try:
            for target in validated_targets:
                result = mount.move_absolute(target, timeout=120)
                row = {"target_deg": target, **asdict(result)}
                sequence_results.append(row)
                print(row)
                # Insert a measurement here only after validating apparatus settling needs.
        except MotionError as error:
            report_motion_error(error)
            raise
else:
    print("Sequence skipped; it requires connection opt-in and the exact SEQUENCE confirmation.")

Clear the confirmation after use. Position readback is the controller's step counter,
not an independent angle measurement. Offline tests and successful notebook execution do
not replace bench acceptance for physical direction, route, settling, and accuracy.
